# `StreamingStdOutCallbackHandler`

Callback handler that writes newly generated LLM tokens directly to standard output while a model response is streaming.

Only the `on_llm_new_token` callback performs output. The other lifecycle callbacks are implemented as no-ops, allowing this handler to display streamed model content without printing chain, tool, agent, start, end, or error messages.

> **Warning**
>
> This handler works only with language models and invocation methods that support streaming.

- Bases: `BaseCallbackHandler`

## Constructor

```python
StreamingStdOutCallbackHandler()
```

The class does not define a custom constructor. It uses the constructor inherited from `BaseCallbackHandler`.

It does not accept a stream object, colour, prefix, suffix, or formatting option in this module.

## Attributes

This class defines no additional instance attributes.

Output is always written to:

```python
sys.stdout
```

## Methods

1. `on_llm_start`: Runs when a non-chat LLM starts.
   * This method is intentionally a no-op.
   * It does not print the serialized model or prompts.
   - **Syntax:**
     ```python
     on_llm_start(
         self,
         serialized: dict[str, Any], # Serialized LLM information
         prompts: list[str], # Prompts submitted to the LLM
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

2. `on_chat_model_start`: Runs when a chat model starts.
   * This method is intentionally a no-op.
   * It does not print the serialized model or input messages.
   - **Syntax:**
     ```python
     on_chat_model_start(
         self,
         serialized: dict[str, Any], # Serialized chat-model information
         messages: list[list[BaseMessage]], # Message batches sent to the model
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

3. `on_llm_new_token`: Writes each newly generated token to standard output.
   * This is the only method in the class that performs output.
   * Converts the supplied token to a string.
   * Writes it without automatically adding a newline.
   * Flushes `sys.stdout` immediately so the token appears as soon as it is received.
   * Available only when model streaming is enabled.
   - **Syntax:**
     ```python
     on_llm_new_token(
         self,
         token: str | list[
             str | dict[str, Any]
         ], # New token or content-block list
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```
   - **Implementation:**
     ```python
     sys.stdout.write(
         str(token)
     )
     sys.stdout.flush()
     ```

4. `on_llm_end`: Runs when model generation finishes successfully.
   * This method is intentionally a no-op.
   * It does not print the final `LLMResult`.
   * It does not add a newline after the streamed response.
   - **Syntax:**
     ```python
     on_llm_end(
         self,
         response: LLMResult, # Final model result
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

5. `on_llm_error`: Runs when model generation fails.
   * This method is intentionally a no-op.
   * It does not print or re-raise the error.
   - **Syntax:**
     ```python
     on_llm_error(
         self,
         error: BaseException, # Model-generation error
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

6. `on_chain_start`: Runs when a chain starts.
   * This method is intentionally a no-op.
   * It does not print chain information or inputs.
   - **Syntax:**
     ```python
     on_chain_start(
         self,
         serialized: dict[str, Any], # Serialized chain information
         inputs: dict[str, Any], # Inputs supplied to the chain
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

7. `on_chain_end`: Runs when a chain finishes successfully.
   * This method is intentionally a no-op.
   * It does not print chain outputs.
   - **Syntax:**
     ```python
     on_chain_end(
         self,
         outputs: dict[str, Any], # Outputs produced by the chain
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

8. `on_chain_error`: Runs when a chain fails.
   * This method is intentionally a no-op.
   * It does not print or re-raise the chain error.
   - **Syntax:**
     ```python
     on_chain_error(
         self,
         error: BaseException, # Chain-execution error
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

9. `on_tool_start`: Runs when a tool starts.
   * This method is intentionally a no-op.
   * It does not print the tool name or input.
   - **Syntax:**
     ```python
     on_tool_start(
         self,
         serialized: dict[str, Any], # Serialized tool information
         input_str: str, # Tool input represented as text
         **kwargs: Any # Additional callback arguments
     ) -> None
     ```

10. `on_agent_action`: Runs when an agent selects an action.
    * This method is intentionally a no-op.
    * It does not print the action log.
    - **Syntax:**
      ```python
      on_agent_action(
          self,
          action: AgentAction, # Selected agent action
          **kwargs: Any # Additional callback arguments
      ) -> Any
      ```

11. `on_tool_end`: Runs when a tool finishes successfully.
    * This method is intentionally a no-op.
    * It does not print the tool output.
    - **Syntax:**
      ```python
      on_tool_end(
          self,
          output: Any, # Tool result
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

12. `on_tool_error`: Runs when a tool fails.
    * This method is intentionally a no-op.
    * It does not print or re-raise the tool error.
    - **Syntax:**
      ```python
      on_tool_error(
          self,
          error: BaseException, # Tool-execution error
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

13. `on_text`: Runs when arbitrary callback text is emitted.
    * This method is intentionally a no-op.
    * Unlike `StdOutCallbackHandler`, it does not print the supplied text.
    - **Syntax:**
      ```python
      on_text(
          self,
          text: str, # Arbitrary callback text
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

14. `on_agent_finish`: Runs when an agent finishes.
    * This method is intentionally a no-op.
    * It does not print the final agent log or return values.
    - **Syntax:**
      ```python
      on_agent_finish(
          self,
          finish: AgentFinish, # Final agent result
          **kwargs: Any # Additional callback arguments
      ) -> None
      ```

## Streaming Behaviour

For a model that emits these tokens:

```text
"Hello"
", "
"world"
"!"
```

the handler performs:

```python
sys.stdout.write("Hello")
sys.stdout.write(", ")
sys.stdout.write("world")
sys.stdout.write("!")
```

The visible result is:

```text
Hello, world!
```

No newline is inserted automatically.

## Immediate Flushing

After every token, the handler calls:

```python
sys.stdout.flush()
```

Without flushing, some environments may buffer output and display several tokens together. Immediate flushing makes the response appear incrementally.

## Content-Block Lists

The `token` parameter may also be:

```python
list[
    str | dict[str, Any]
]
```

The handler does not process each content block separately. It applies:

```python
str(token)
```

to the complete value.

For example:

```python
token = [
    {
        "type": "text",
        "text": "Hello"
    }
]
```

may be printed using its Python list-and-dictionary representation:

```text
[{'type': 'text', 'text': 'Hello'}]
```

## No Automatic Final Newline

`on_llm_end` is a no-op, so the handler does not add a newline after streaming completes.

A caller that requires a newline must print one separately:

```python
print()
```

## Comparison with `StdOutCallbackHandler`

| Behaviour | `StreamingStdOutCallbackHandler` | `StdOutCallbackHandler` |
|---|---|---|
| Prints each streamed token | Yes | No dedicated token implementation in `stdout.py` |
| Flushes after each token | Yes | Not applicable |
| Prints chain start/end | No | Yes |
| Prints tool output | No | Yes |
| Prints agent logs | No | Yes |
| Prints arbitrary `on_text` values | No | Yes |
| Supports configured colour | No | Yes |
| Adds a final newline automatically | No | Depends on callback method |

## Example

```python
from langchain_core.callbacks.streaming_stdout import (
    StreamingStdOutCallbackHandler,
)

handler = StreamingStdOutCallbackHandler()
```

Use the handler with a streaming-capable model:

```python
model = chat_model.bind(
    callbacks=[
        StreamingStdOutCallbackHandler()
    ]
)
```

As tokens are generated, they are written directly to the terminal.

## Direct Method Example

```python
handler = StreamingStdOutCallbackHandler()

handler.on_llm_new_token(
    "Hello"
)
handler.on_llm_new_token(
    ", world!"
)

print()
```

Output:

```text
Hello, world!
```

## Important Notes

* The handler does not enable streaming by itself.
* The model and invocation path must already support streaming callbacks.
* It writes to the process-wide `sys.stdout`.
* It does not provide a way to redirect output to another stream.
* It performs no token formatting.
* It performs no colour formatting.
* It does not print errors.
* It does not add separators between tokens.
* It does not add a newline when generation ends.

## Source

This reference follows the pinned LangChain source:

```text
libs/core/langchain_core/callbacks/streaming_stdout.py
Commit: 1c3a4186cf2ba4f28face59118ac7786de009f91
```